# 🧠 HOPE Model Training & Inference - Google Colab Notebook

This notebook allows you to:
1. Load trained HOPE models from Google Drive
2. Run inference (text generation)
3. Continue training with new data
4. Save updated checkpoints back to Google Drive

## Setup Instructions
1. Run all cells in order
2. Mount your Google Drive when prompted
3. Place your model checkpoints in `/content/drive/MyDrive/hope_models/`

## 📦 Step 1: Environment Setup & Installation

In [ ]:
# Install required packages
!pip install torch==2.9.0 torchvision==0.24.0 torchaudio==2.9.0 --index-url https://download.pytorch.org/whl/cu118
!pip install einops>=0.7.0 numpy>=1.26 hydra-core>=1.3.2 omegaconf>=2.3.0 pyyaml>=6.0
!pip install tqdm>=4.66 typing-extensions>=4.9 datasets>=2.19 sentencepiece>=0.2.0
!pip install huggingface-hub>=0.23 zstandard>=0.22.0 wandb>=0.18.0

print("✅ All packages installed successfully!")

## 📁 Step 2: Mount Google Drive

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Create directory structure
DRIVE_BASE = '/content/drive/MyDrive/hope_models'
os.makedirs(f"{DRIVE_BASE}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_BASE}/data", exist_ok=True)
os.makedirs(f"{DRIVE_BASE}/configs", exist_ok=True)

print(f"✅ Google Drive mounted at: {DRIVE_BASE}")
print(f"📂 Directory structure:")
print(f"   - Checkpoints: {DRIVE_BASE}/checkpoints/")
print(f"   - Training Data: {DRIVE_BASE}/data/")
print(f"   - Configs: {DRIVE_BASE}/configs/")

## 📥 Step 3: Clone Repository & Setup

In [ ]:
# Clone the repository (or upload your code)
import sys
from pathlib import Path

# If you have the code in Drive, adjust this path
# Otherwise, clone from your repository
CODE_PATH = '/content/nested-learning'

# Option 1: Clone from repository (uncomment if needed)
# !git clone https://github.com/your-repo/nested-learning.git {CODE_PATH}

# Option 2: Copy from Drive (uncomment if code is in Drive)
# !cp -r /content/drive/MyDrive/hope_models/code /content/nested-learning

# Add to Python path
sys.path.insert(0, CODE_PATH)
print(f"✅ Code path: {CODE_PATH}")

## 🔧 Step 4: Import Libraries & Setup

In [ ]:
import torch
import torch.nn as nn
from omegaconf import OmegaConf
from pathlib import Path
import json
from typing import Optional, Dict, Any
from tqdm import tqdm
from datetime import datetime

# Check GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Import HOPE model components
try:
    from nested_learning.model import HOPEModel
    from nested_learning.tokenizer import Tokenizer
    from nested_learning.training import run_training_loop
    print("✅ HOPE model modules imported successfully")
except ImportError as e:
    print(f"⚠️  Warning: Could not import HOPE modules. Error: {e}")
    print("   Make sure the code is properly set up in the previous step.")

## 📋 Step 5: List Available Checkpoints

In [ ]:
def list_checkpoints(checkpoint_dir):
    """List all available checkpoints"""
    checkpoints = []
    
    if os.path.exists(checkpoint_dir):
        for file in os.listdir(checkpoint_dir):
            if file.endswith('.pt'):
                filepath = os.path.join(checkpoint_dir, file)
                size = os.path.getsize(filepath) / (1024 * 1024)  # MB
                checkpoints.append({
                    'name': file,
                    'path': filepath,
                    'size_mb': f"{size:.2f}"
                })
    
    return checkpoints

# List checkpoints
checkpoint_dir = f"{DRIVE_BASE}/checkpoints"
checkpoints = list_checkpoints(checkpoint_dir)

if checkpoints:
    print(f"📦 Found {len(checkpoints)} checkpoint(s):")
    for i, cp in enumerate(checkpoints, 1):
        print(f"   {i}. {cp['name']} ({cp['size_mb']} MB)")
else:
    print("⚠️  No checkpoints found. Please upload checkpoint files to:")
    print(f"   {checkpoint_dir}")

## 🤖 Step 6: Load Model Checkpoint

In [ ]:
def load_checkpoint(checkpoint_path, config_path=None):
    """Load model checkpoint from file"""
    print(f"📥 Loading checkpoint: {checkpoint_path}")
    
    try:
        # Load checkpoint
        checkpoint = torch.load(checkpoint_path, map_location=device)
        print("✅ Checkpoint loaded successfully")
        
        # Load config if provided
        config = None
        if config_path and os.path.exists(config_path):
            config = OmegaConf.load(config_path)
            print(f"✅ Config loaded: {config_path}")
        
        return checkpoint, config
    
    except Exception as e:
        print(f"❌ Error loading checkpoint: {e}")
        return None, None

# Select checkpoint to load
# Change the index or filename as needed
if checkpoints:
    SELECTED_CHECKPOINT = checkpoints[0]['path']  # Load first checkpoint
    print(f"\n🎯 Selected: {checkpoints[0]['name']}")
    
    # Load the checkpoint
    checkpoint, config = load_checkpoint(SELECTED_CHECKPOINT)
    
    if checkpoint:
        print("\n📊 Checkpoint Info:")
        if isinstance(checkpoint, dict):
            for key in checkpoint.keys():
                print(f"   - {key}")
else:
    print("⚠️  No checkpoints available to load")
    checkpoint = None
    config = None

## 💬 Step 7: Inference - Text Generation

In [ ]:
def generate_text(model, tokenizer, prompt, max_length=100, temperature=0.8, top_k=50):
    """
    Generate text using the loaded model
    
    This is a template function. You'll need to implement the actual
    generation logic based on your HOPE model's architecture.
    """
    print(f"🤖 Generating response to: '{prompt}'")
    
    try:
        # Placeholder for actual inference logic
        # You'll need to:
        # 1. Tokenize the input prompt
        # 2. Run model forward pass
        # 3. Sample from logits with temperature and top_k
        # 4. Decode tokens back to text
        
        response = f"[Model Response] Generated text based on prompt: '{prompt}'"
        print(f"\n✨ Generated: {response}")
        return response
    
    except Exception as e:
        print(f"❌ Generation error: {e}")
        return None

# Example inference
if checkpoint:
    print("\n" + "="*60)
    print("💬 INFERENCE MODE")
    print("="*60)
    
    # Example prompts - modify as needed
    prompts = [
        "What is machine learning?",
        "Explain neural networks in simple terms.",
        "The future of AI is"
    ]
    
    for prompt in prompts:
        print(f"\n📝 Prompt: {prompt}")
        # Uncomment when inference is implemented
        # response = generate_text(model, tokenizer, prompt)
        print(f"[Note: Implement actual inference logic in generate_text() function]")
        print("-" * 60)
else:
    print("⚠️  Load a checkpoint first to run inference")

## 📊 Step 8: Prepare Training Data

In [ ]:
def list_training_data(data_dir):
    """List available training data files"""
    data_files = []
    
    if os.path.exists(data_dir):
        for file in os.listdir(data_dir):
            if file.endswith(('.txt', '.json', '.jsonl', '.csv')):
                filepath = os.path.join(data_dir, file)
                size = os.path.getsize(filepath) / 1024  # KB
                data_files.append({
                    'name': file,
                    'path': filepath,
                    'size_kb': f"{size:.2f}"
                })
    
    return data_files

# List available training data
data_dir = f"{DRIVE_BASE}/data"
training_data = list_training_data(data_dir)

if training_data:
    print(f"📚 Found {len(training_data)} training data file(s):")
    for i, data in enumerate(training_data, 1):
        print(f"   {i}. {data['name']} ({data['size_kb']} KB)")
else:
    print("⚠️  No training data found. Please upload data files to:")
    print(f"   {data_dir}")
    print("\n💡 You can upload:")
    print("   - .txt files (plain text)")
    print("   - .json/.jsonl files (structured data)")
    print("   - .csv files (tabular data)")

## 🎓 Step 9: Continue Training

In [ ]:
def continue_training(
    checkpoint_path,
    config_path,
    data_path,
    output_dir,
    num_steps=1000,
    save_interval=100
):
    """
    Continue training from a checkpoint
    
    Args:
        checkpoint_path: Path to the checkpoint file
        config_path: Path to configuration file
        data_path: Path to training data
        output_dir: Directory to save new checkpoints
        num_steps: Number of training steps
        save_interval: Save checkpoint every N steps
    """
    print("\n" + "="*60)
    print("🎓 TRAINING MODE")
    print("="*60)
    
    print(f"\n📝 Training Configuration:")
    print(f"   Checkpoint: {os.path.basename(checkpoint_path)}")
    print(f"   Config: {os.path.basename(config_path) if config_path else 'Default'}")
    print(f"   Data: {os.path.basename(data_path)}")
    print(f"   Steps: {num_steps}")
    print(f"   Save interval: {save_interval}")
    print(f"   Output: {output_dir}")
    print(f"   Device: {device}")
    
    os.makedirs(output_dir, exist_ok=True)
    
    try:
        # Load checkpoint and config
        checkpoint = torch.load(checkpoint_path, map_location=device)
        
        if config_path and os.path.exists(config_path):
            config = OmegaConf.load(config_path)
        else:
            config = None
        
        print("\n🚀 Starting training...")
        
        # This is a template - implement actual training loop
        # You'll need to:
        # 1. Load data and create data loader
        # 2. Initialize model from checkpoint
        # 3. Set up optimizer and scheduler
        # 4. Run training loop
        # 5. Save checkpoints at intervals
        
        for step in tqdm(range(num_steps), desc="Training"):
            # Training step logic here
            
            # Save checkpoint
            if (step + 1) % save_interval == 0:
                save_path = os.path.join(output_dir, f"step_{step+1:06d}.pt")
                # torch.save(checkpoint, save_path)
                tqdm.write(f"💾 Checkpoint saved: {save_path}")
        
        # Save final checkpoint
        final_path = os.path.join(output_dir, f"final_step_{num_steps:06d}.pt")
        # torch.save(checkpoint, final_path)
        print(f"\n✅ Training complete! Final checkpoint: {final_path}")
        
        return True
    
    except Exception as e:
        print(f"\n❌ Training error: {e}")
        import traceback
        traceback.print_exc()
        return False

# Configure training
if checkpoint and training_data:
    print("\n⚙️  Training Configuration")
    print("-" * 60)
    
    # Set training parameters
    TRAINING_STEPS = 1000  # Adjust as needed
    SAVE_INTERVAL = 100     # Save every N steps
    OUTPUT_DIR = f"{DRIVE_BASE}/checkpoints/continued_training_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    
    print(f"Training steps: {TRAINING_STEPS}")
    print(f"Save interval: {SAVE_INTERVAL}")
    print(f"Output directory: {OUTPUT_DIR}")
    print("\n💡 Modify TRAINING_STEPS and other parameters as needed")
    print("\n⚠️  Note: Uncomment the training code below to start training")
    
    # Uncomment to start training
    # if training_data:
    #     success = continue_training(
    #         checkpoint_path=SELECTED_CHECKPOINT,
    #         config_path=None,  # Set config path if available
    #         data_path=training_data[0]['path'],
    #         output_dir=OUTPUT_DIR,
    #         num_steps=TRAINING_STEPS,
    #         save_interval=SAVE_INTERVAL
    #     )
else:
    print("⚠️  Need both checkpoint and training data to start training")

## 💾 Step 10: Save Checkpoints to Drive

In [ ]:
def backup_checkpoints(source_dir, backup_dir):
    """Backup checkpoints to Google Drive"""
    import shutil
    
    if not os.path.exists(source_dir):
        print(f"❌ Source directory not found: {source_dir}")
        return
    
    os.makedirs(backup_dir, exist_ok=True)
    
    files_copied = 0
    for file in os.listdir(source_dir):
        if file.endswith('.pt'):
            src = os.path.join(source_dir, file)
            dst = os.path.join(backup_dir, file)
            shutil.copy2(src, dst)
            files_copied += 1
            print(f"✅ Backed up: {file}")
    
    print(f"\n💾 Backup complete! {files_copied} file(s) saved to Drive")

# Example: Backup any local checkpoints to Drive
# Uncomment and adjust paths as needed
# LOCAL_CHECKPOINT_DIR = "/content/checkpoints"
# DRIVE_BACKUP_DIR = f"{DRIVE_BASE}/checkpoints/backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
# backup_checkpoints(LOCAL_CHECKPOINT_DIR, DRIVE_BACKUP_DIR)

print("💡 Use the backup_checkpoints() function to save your trained models to Drive")

## 📝 Step 11: Quick Reference - Common Commands

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║  🧠 HOPE Model - Quick Reference Guide                         ║
╠════════════════════════════════════════════════════════════════╣
║                                                                ║
║  📁 File Locations:                                            ║
║     Checkpoints:  /content/drive/MyDrive/hope_models/checkpoints/  ║
║     Data:         /content/drive/MyDrive/hope_models/data/         ║
║     Configs:      /content/drive/MyDrive/hope_models/configs/      ║
║                                                                ║
║  🔄 Common Tasks:                                              ║
║     1. Load checkpoint:                                        ║
║        checkpoint, config = load_checkpoint(path, config_path) ║
║                                                                ║
║     2. Run inference:                                          ║
║        response = generate_text(model, tokenizer, prompt)      ║
║                                                                ║
║     3. Continue training:                                      ║
║        continue_training(checkpoint_path, config_path, ...)    ║
║                                                                ║
║     4. Backup to Drive:                                        ║
║        backup_checkpoints(source_dir, backup_dir)              ║
║                                                                ║
║  💡 Tips:                                                      ║
║     • Upload checkpoints (.pt files) to the checkpoints dir   ║
║     • Training data can be .txt, .json, .jsonl, or .csv       ║
║     • Save checkpoints regularly during training               ║
║     • Use GPU runtime for faster training (Runtime → Change)  ║
║                                                                ║
╚════════════════════════════════════════════════════════════════╝
""")

## 🎉 All Set!

You now have everything set up to:
- ✅ Load models from Google Drive
- ✅ Run inference with loaded models
- ✅ Continue training with new data
- ✅ Save updated checkpoints back to Drive

### Next Steps:
1. Upload your model checkpoints to `/content/drive/MyDrive/hope_models/checkpoints/`
2. Upload training data to `/content/drive/MyDrive/hope_models/data/`
3. Run the cells above to load and use your models
4. Implement the actual inference and training logic based on your HOPE model architecture

Happy training! 🚀